# UK House Price Analysis & Prediction (2015–2024)

## Project Notebook

<a id="introduction"></a>
## Introduction
This notebook explores trends and drivers of UK residential property prices between 2015 and 2024 using the **UK House Price Prediction dataset**, sourced from [Kaggle](https://www.kaggle.com/datasets/swarupsudulaganti/uk-house-price-prediction-dataset-2015-to-2024).
The goal is to explore historical patterns, identify regional disparities, and model potential future price trends.

The dataset includes:
- Sale price and date of transaction  
- Property address details (postcode, street, locality, town, district, county)  
- Property type (Detached, Semi-detached, etc.)  
- Whether the property is a new build  
- Tenure (Freehold/Leasehold)  

In addition, this project incorporates:
- A **custom mapping file** with UK region coordinates for geographical visualisations  
- A **GeoJSON file** for regional mapping of data (ONS)

**Analytical approach:**  
The project combines exploratory data analysis, geospatial visualisation (2D and 3D maps), and predictive modelling. Interactive charts and maps are produced using libraries such as Plotly, PyDeck, and GeoPandas.

### Research Questions

1. Which UK region has the highest and lowest average house prices?
2. Is the UK housing market stable or volatile?
3. What are the future house price trends in the North West region?
4. How have UK property prices changed over time?
5. Are there notable regional differences (e.g., North vs. South)?

## **Table of Contents**

### [Introduction](#introduction)

### [1. Data Import and Cleaning](#1-data-import-and-cleaning)
- [1.1 Data Import and Setup](#11-data-import-and-setup)  
- [1.2 Inspect & Understand the Dataset](#12-inspect--understand-the-dataset)  
- [1.3 Data Cleaning](#13-data-cleaning)  
- [1.4 County Name Standardisation with Mapping File](#14-county-name-standardisation-with-mapping-file)  
- [1.5 Outlier Investigation & Property Value Distribution](#15-outlier-investigation--property-value-distribution)  
- [1.6 Final Cleaning: Remove Irrelevant & Invalid Entries](#16-final-cleaning-remove-irrelevant--invalid-entries)  

### [2. Exploratory Data Analysis](#2-exploratory-data-analysis)
- [2.1 Geographic Analysis (Macro Level)](#21-geographic-analysis-macro-level)  
- [2.2 Distributions & Group Comparisons (Mid-Level)](#22-distributions--group-comparisons-mid-level)  
- [2.3 Time Series Analysis (Fine Detail)](#23-time-series-analysis-fine-detail)  

### [3. Statistical Analysis and Visualisations](#3-statistical-analysis-and-visualisations)
- [3.1 Pricing Forecast](#31-pricing-forecast)  
- [3.2 Modelling Decisions and SARIMA Forecasting](#32-modelling-decisions-and-sarima-forecasting)  
- [3.3 Comparing ARIMA and SARIMA Forecasting Models](#33-comparing-arima-and-sarima-forecasting-models)  
- [3.4 Summary of Time Series Analysis](#34-summary-of-time-series-analysis)  
- [3.5 Regional Analysis - North vs. South](#35-regional-analysis---north-vs-south)  
- [3.6 Regional Price Changes](#36-regional-price-changes)  
- [3.7 T-Test: North vs. South Property Prices](#37-t-test-north-vs-south-property-prices)  

### [4. List of Tables and Figures](#4-list-of-tables-and-figures)

<a id="1-data-import-and-cleaning"></a>
## 1. Data Import and Cleaning

This section includes:
- Required Python libraries for data analysis, statistics, and machine learning
- Data importation and initial configuration
- Setup for consistent styling and performance

<a id="11-data-import-and-setup"></a>
### 1.1 Data Import and Setup

In [ ]:
# %pip install missingno pydeck
# %pip install geopandas pydeck
# %pip install fuzzywuzzy python-Levenshtein

In [ ]:
import os

# Use your specific path
path = r"C:\Users\Parkf\Documents\Project"

# GITHUB: When you upload, you would uncomment the line below 
# and comment out your local path above.
# path = os.getcwd() 

os.chdir(path)
print(f"✅ Project Root set to: {os.getcwd()}")

In [ ]:
# Core Python Libraries
import datetime  # Handling date and time
import re        # Regular expressions for pattern matching
import warnings  # Suppressing warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

# Data Manipulation
import numpy as np             # Numerical operations with arrays and matrices
import pandas as pd            # Data analysis and manipulation with DataFrames

# Data Visualization
import matplotlib.pyplot as plt        # Basic plotting
import seaborn as sns                  # Statistical data visualisation
import plotly.express as px            # Interactive plots 
import plotly.graph_objects as go      # Interactive plots 
import missingno as msno               # Visualising missing data
import pydeck as pdk                   # Geospatial mapping and visualisation
import json                            # Allows json file import
import geopandas as gpd                # Mapping

# Set default plot styles
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')

# Interactive Widgets and Display
import ipywidgets as widgets                   # Interactive Jupyter widgets
from IPython.display import display, HTML      # Display tools for better outputs in notebooks

# Machine Learning and Preprocessing (Scikit-learn) 
from sklearn.model_selection import train_test_split      # Train/test split
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # Feature scaling & encoding
from sklearn.impute import SimpleImputer                  # Handling missing values
from sklearn.compose import ColumnTransformer             # Combining preprocessing steps
from sklearn.pipeline import Pipeline                     # Building ML pipelines
from sklearn.linear_model import LinearRegression         # Linear regression model
from sklearn.ensemble import RandomForestRegressor        # Ensemble model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Model evaluation

# Statistical Modeling (Statsmodels)
import statsmodels.api as sm                          # Core statsmodels API
import statsmodels.formula.api as smf                 # Formula API for modeling
from statsmodels.tsa.seasonal import seasonal_decompose  # Decompose time series
from statsmodels.tsa.stattools import adfuller           # Augmented Dickey-Fuller test
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf  # ACF and PACF plots
from statsmodels.tsa.arima.model import ARIMA              # ARIMA model
from statsmodels.tsa.statespace.sarimax import SARIMAX     # SARIMAX model
from statsmodels.stats.multicomp import pairwise_tukeyhsd  # Tukey

# Scientific Computing (SciPy) 
import scipy                         # Scientific functions 
from scipy import stats              # Statistical functions 

# Utilities 
import requests              # Making HTTP requests / fetching APIs
from fuzzywuzzy import process  # Fuzzy string matching
from tqdm import tqdm           # Progress bars

In [ ]:
# Load the dataset into a dataframe
df = pd.read_csv("UK_House_Price_Prediction_dataset_2015_to_2024.csv")

<a id="12-inspect--understand-the-dataset"></a>
### 1.2 Inspect & Understand the Dataset

In [ ]:
# Initial dataset inspection
df.head()

In [ ]:
print(df.info())

In [ ]:
print(df.describe(include='all')) # includes all categorical information

**Note:** The column headers make sense with dates in date, post codes in post code numbers in price etc. The data seems well formatted from this initial inspection so I will move on to checking for missing values and outliers.

<a id="13-data-cleaning"></a>
### 1.3 Data Cleaning

In [ ]:
# Missing values check
print(df.isnull().sum())
print("Total missing values in the dataset:", df.isnull().sum().sum())

**Note:** No missing values

In [ ]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()
print("Duplicates removed. New shape:", df.shape)

In [ ]:
# Make sure date columns are in datetime format
df['date'] = pd.to_datetime(df['date'].str.strip(), dayfirst=True, errors='coerce')

**Note:** Date strings were converted to `datetime` objects to enable time-series analysis.

In [ ]:
# Check for inconsistencies with names for the location columns
# Standardise the text format
df['town'] = df['town'].str.strip().str.lower()
df['district'] = df['district'].str.strip().str.lower()
df['county'] = df['county'].str.strip().str.lower()

In [ ]:
# Check how many unique values there are
print("Unique towns:", df['town'].nunique())
print("Unique districts:", df['district'].nunique())
print("Unique counties:", df['county'].nunique())

In [ ]:
def clean_location(x):
    if pd.isnull(x):
        return x
    return x.strip().title()

df['town'] = df['town'].apply(clean_location)
df['district'] = df['district'].apply(clean_location)
df['county'] = df['county'].apply(clean_location)

**Rationale:** Lowercasing and stripping whitespace helps ensure uniformity across region names.

In [ ]:
# Check number of unique values
print("Unique towns:", df['town'].nunique())
print("Unique districts:", df['district'].nunique())
print("Unique counties:", df['county'].nunique())

 **Note:** No difference from the previous check so I will check the names themselves. There are too many to manually check so I will use fuzzy matching to identify potential duplicates. I won't be going down to town or district level so I will check counties.

<a id="14-county-name-standardisation-with-mapping-file"></a>
### 1.4 County Name Standardisation with Mapping File
To eliminate inconsistencies in the `county` field caused by formatting errors or varying naming conventions (e.g. `"City of Nottingham"` vs `"Nottinghamshire"`), I used a **custom mapping file** (`county_mapping.csv`). This file maps incorrect or ambiguous county names to their correct, standardised counterparts.

This step ensures:
- Accurate geographic aggregation in later analyses (especially maps)
- Reduction of duplication due to slight naming variations
- A more realistic regional distribution of property data

In [ ]:
# Normalise County Names
df['county'] = df['county'].str.strip().str.title()

In [ ]:
# Get all unique county names
unique_counties = df['county'].dropna().unique()

# Build fuzzy matches for all counties
similar_county_matches = {}
for county in unique_counties:
    matches = process.extract(county, unique_counties, limit=5)
    similar_county_matches[county] = matches

In [ ]:
for county, matches in similar_county_matches.items():
    print(f"{county}: {matches}")

In [ ]:
# Load and Apply County Mapping
mapping_df = pd.read_csv(r'C:\Users\Parkf\Documents\Project\county_mapping.csv')
df_corrected = df.merge(mapping_df, on='county', how='left')

# 1. Merge (Keep 'county' as the join key)
df_corrected = df.merge(mapping_df, on='county', how='left')

# 2. Use the correct capitalisation: 'Replacement_county'
df_corrected['county'] = df_corrected['Replacement_county'].combine_first(df_corrected['county'])

# 3. Drop the column using the correct capitalisation
df_corrected.drop(columns=['Replacement_county'], inplace=True)

# 4. Drop the rest
df_corrected.drop(columns=['street', 'locality', 'district'], inplace=True, errors='ignore')

### North/South Region Indicator

To support regional analysis later in this project, I have created a new column indicating whether a property is in the **North** or **South** of the UK.

While the Severn-Wash Line is commonly used in geographical discourse to divide the country, I have decided not to use it here. Instead, my classification is based on a simpler rule: the Midlands and all counties further north are classified as ‘North’, and everything below that as ‘South’. This split roughly divides the dataset into two balanced geographic halves, making it more suitable for comparative analysis of price and income trends.

This North/South label is merged in using the county mapping file and will be used in later visualisations and modelling.

In [ ]:
# View the result
df_corrected.head()

In [ ]:
# Check the changes have been made
print("Unique counties:", df_corrected['county'].nunique())

In [ ]:
# View all counties and counts of each
county_counts = df_corrected['county'].value_counts().sort_index()
print(county_counts.to_string()) # shows all rows

#### Final Count of Properties by Standardised County Name

After applying the mapping, the number of unique counties was reduced from **117** to **69**, confirming successful consolidation of duplicated or inconsistent names.

> **Insight:** A manual scan confirms that the resulting county names align with official UK naming conventions. This cleanup enables accurate region level visualisations and analysis going forward.


### Section Summary: Location Cleanup & Mapping

- Used `county_mapping.csv` to fix name inconsistencies
- Reduced county categories from 117 → 69 (41% reduction)
- Removed redundant location columns (`street`, `locality`, `district`)
- Confirmed standardisation through manual validation

The dataset is now geographically coherent and ready for visualisation and group-based analysis (e.g., regional choropleth maps, time trends by county).


<a id="15-outlier-investigation--property-value-distribution"></a>
### 1.5 Outlier Investigation & Property Value Distribution

This section explores price distributions across all property types and identifies outliers that may distort analysis. Boxplots are used to visualise spread and detect anomalies. Data tables highlight the highest and lowest valued properties by category.


In [ ]:
# Overall price distribution (Boxplot)
plt.figure(figsize=(12, 4)) 
sns.boxplot(x=df_corrected['price'])
plt.title("Boxplot of All Property Prices")
plt.xlabel("Price")
plt.show()

#### Figure 1: Overall Distribution of UK Property Prices

This boxplot shows the full distribution of property prices, including extreme high-value properties.


In [ ]:
# Mapping of property type codes to descriptive labels
property_type_labels = {
    'D': 'Detached',
    'F': 'Flats',
    'O': 'Offices',
    'S': 'Semi-detached',
    'T': 'Terraced'
}

# Map the labels
df_corrected['property_type_label'] = df_corrected['property_type'].map(property_type_labels)

In [ ]:
# Boxplots for each property type
plt.figure(figsize=(10, 6))
sns.boxplot(x='property_type_label', y='price', data=df_corrected)
plt.xlabel("Property Type")
plt.ylabel("Price")
plt.title("Price Distribution by Property Type")
plt.xticks(rotation=45)  
plt.tight_layout()
plt.show()

#### Figure 2: Price Distribution by Property Type

Offices show extreme outliers and a wide range in values.


In [ ]:
# Create a boxplot for each property_type using a reusable function
def plot_boxplot(df_corrected, property_type_code, column='price'):
    """
    Plots a boxplot for a given property type

    Args:
        df (DataFrame): The dataset.
        property_type_code (str): The code for the property type ('D', 'F', 'O', 'S', or 'T').
        column (str): The column to plot (default is 'price').
    """
    # Filter dataframe
    df_filtered = df_corrected[df_corrected['property_type'] == property_type_code]
    
    # Check the filtered dataframe is not empty
    if df_filtered.empty:
        print(f"No data for property type '{property_type_code}'.")
        return
    
    # Get the label from the mapping
    property_label = property_type_labels.get(property_type_code, 'Unknown')

    # Plot
    plt.figure(figsize=(12, 3))
    sns.boxplot(x=df_filtered[column])
    plt.title(f"Box plot of {column} for Property Type: {property_label}")
    plt.xlabel(column)
    plt.show()

In [ ]:
# Create mutliple boxplots
for code in ['D', 'F', 'O', 'S', 'T']:
    plot_boxplot(df, code)

#### Figure 3: Boxplots by Individual Property Type

Each panel shows the distribution of prices for a specific property category. Office prices are massively different to the other property types as its scale is 100x greater so I will create a dataset purely made up of residential properties.

### 1.5.1  Top 5 Highest Valued Properties (by Type)

The following tables show the top five most expensive properties for each residential category. Displayed side-by-side for ease of comparison.


In [ ]:
# Horizontal HTML display
property_codes = ['D', 'F', 'S', 'T']

# Container for all HTML blocks
html_blocks = []

for code in property_codes:
    property_label = property_type_labels.get(code, 'Unknown')
    
    # Filter and get top 5
    df_filtered = df_corrected[df_corrected['property_type'] == code]
    top5 = df_filtered.sort_values(by='price', ascending=False).head(5)
    
    # Convert top5 table to HTML
    table_html = top5[['date', 'price', 'town', 'region']].to_html(index=False)
    
    # Add a titled block with CSS styling for fixed width & inline block display
    block_html = f"""
    <div style="display:inline-block; vertical-align:top; margin-right:20px; width:30%;">
        <h3>Top 5 Most Valuable Properties - {property_label}</h3>
        {table_html}
    </div>
    """
    html_blocks.append(block_html)

# Join blocks in one HTML string and display
display(HTML("".join(html_blocks)))

#### Table 1: Top 5 Most Expensive Properties by Type

London or South East regions dominates high-end property values across all residential categories. These results validate the outliers seen in earlier boxplots and represent real market values rather than data errors.


### 1.5.2 Bottom 5 Least Valuable Properties (by Type)

The following output highlights the cheapest properties sold in each category. Some values are suspiciously low and warrant further inspection.


In [ ]:
# Horizontal HTML display
property_codes = ['D', 'F', 'S', 'T']

# Container for all HTML blocks
html_blocks = []

for code in property_codes:
    property_label = property_type_labels.get(code, 'Unknown')
    
    # Filter and get bottom 5
    df_filtered = df_corrected[df_corrected['property_type'] == code]
    bottom5 = df_filtered.sort_values(by='price', ascending=True).head(5)
    
    # Convert bottom5 table to HTML
    table_html = bottom5[['date', 'price', 'town', 'region']].to_html(index=False)
    
    # Add a titled block with CSS styling for fixed width & inline-block display
    block_html = f"""
    <div style="display:inline-block; vertical-align:top; margin-right:20px; width:30%;">
        <h3>Bottom 5 Least Valuable Properties - {property_label}</h3>
        {table_html}
    </div>
    """
    html_blocks.append(block_html)

# Join blocks in one HTML string and display
display(HTML("".join(html_blocks)))

#### Table 2: Bottom 5 Least Expensive Properties by Type

Low-end values raise potential concerns about data quality, especially where prices are implausibly low (e.g., under £1,000). 


### Outlier detection - Lower Outliers

In [ ]:
# Function to detect lower outliers only
def detect_lower_outliers(group):
    Q1 = group['price'].quantile(0.25)
    Q3 = group['price'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR

    # Add a column flagging low outliers
    group['is_low_outlier'] = group['price'] < lower_bound
    return group

# Apply across counties
df_flagged = df_filtered.groupby('county', group_keys=False).apply(detect_lower_outliers)

# Display suspiciously low values
df_flagged[df_flagged['is_low_outlier']].sort_values('price').head(20)

#### Table 3: Suspected Low-Value Outliers by County

This table lists the lowest-valued properties flagged using the 1.5×IQR rule, grouped by county. Most low values appear plausible.

In [ ]:
# Manually remove the property at £700 in Warrington
df_filtered = df_filtered[~((df_filtered['price'] == 700) & (df_filtered['town'] == 'Warrington'))]

**Manual Adjustment:**  
Although not flagged by the previous method, a residential property in *Warrington* with a sale price of **£700** was manually removed. Based on context and a review of other local prices, this value is likely incorrect.

In [ ]:
# Check the dataset
df_filtered.sort_values('price').head(10)

#### Table 4: Price check for low values
Even though there are still some very low values, the remaining prices are plausible. Mostly terraced houses in the North which I would expect to be at the lower end of the property valuations.

### Notes:

During outlier investigation and data inspection, a small number of manual adjustments were made to improve data quality:

- **Removed an anomalously low-priced property in Warrington** (£700) that was not flagged by the IQR-based outlier detection method but was clearly implausible on visual/manual review.
  
  > Although it wasn’t statistically an outlier, the price was unreasonably low given the context. Manual removal ensures it does not skew regional averages or modelling outcomes.

- **Excluded commercial properties** (Offices) by filtering out property type `'O'`, focusing the analysis on residential trends only.

These steps ensure the dataset is accurate and not skewed by obviosuly incorrect values.


<a id="16-final-cleaning-remove-irrelevant--invalid-entries"></a>
### 1.6 Final Cleaning: Remove Irrelevant & Invalid Entries

This section filters out non-residential property types and invalid postcodes to ensure that only relevant, clean data is used for downstream analysis.

In [ ]:
# Keep only residential property types
residential_types = ['D', 'F', 'S', 'T']
df_residential = df_corrected[df_corrected['property_type'].isin(residential_types)].copy()

# Show counts by property type
print(df_residential['property_type'].value_counts())
df_residential.head()

**Action Taken:**  
Removed non-residential entries (offices) to focus exclusively on housing trends. 

In [ ]:
# Define function to validate UK postcodes
def is_valid_uk_postcode(postcode):
    pattern = r"^[A-Z]{1,2}[0-9][0-9A-Z]?\s?[0-9][A-Z]{2}$"
    return bool(re.match(pattern, postcode.strip().upper()))

# Apply to dataframe
df_residential['postcode_valid'] = df_residential['postcode'].apply(is_valid_uk_postcode)

# Filter only valid postcodes
df_residential = df_residential[df_residential['postcode_valid']]

# Check property type counts
print(df_residential['property_type'].value_counts())

**Validation Step:**  
UK postcodes were validated using a regex pattern to catch incorrectly formatted entries (e.g., missing digits or stray characters).

> **Result:** All postcodes passed validation. No rows were removed in this step.


## End of Data Preparation

At this point, all relevant data has been:
- Imported and inspected
- Cleaned (duplicates, invalid values, inconsistent formatting)
- Standardised (county names, postcodes)
- Filtered for **residential properties only**

The cleaned and enriched dataset is now ready for:
- Geographical analysis (e.g. choropleth maps)
- Time-series exploration
- Feature engineering and predictive modelling

I will now move into Exploratory Data Analysis (EDA) to uncover patterns and relationships in the data.

<a id="2-exploratory-data-analysis"></a>
## 2. Exploratory Data Analysis

This section explores trends and patterns in the cleaned residential property dataset. It includes geographical, regional, and structural data analyses using various visualisation and summary techniques. As mentioned, I will focus purely on residential properties for the rest of the project.

<a id="21-geographic-analysis-macro-level"></a>
### 2.1 Geographic Analysis (Macro Level)
This section visualises property price distributions geographically across the UK.

### Geocoding Postcodes

To plot properties spatially, postcodes were converted to latitude and longitude coordinates using the [postcodes.io API](https://postcodes.io/). Since API requests are time-consuming, results were cached in a CSV for reuse.

In [ ]:
# Run the below once to create a postcode mapping file

#### Post Code Mapping Summary
The dataset originally contained 85,577 residential property records. After merging with the latitude and longitude data, 85,558 rows were successfully matched with geographic coordinates.   
This means that 99.98% of the residential properties were successfully geocoded, which is excellent coverage. The small number of unmatched postcodes is negligible and does not impact the analyses.

In [ ]:
# 1. Load using the full path to your Project folder
postcode_coords = pd.read_csv(r'postcode_coords.csv')

# 2. Merge coordinates into your residential data
df_geo = df_residential.merge(postcode_coords, on='postcode', how='left')

# 3. Clean up any rows that didn't find a match in the API results
df_geo = df_geo.dropna(subset=['latitude', 'longitude'])

In [ ]:
print(f"Rows in df_residential: {len(df_residential):,}")
print(f"Rows in df_geo (with coordinates): {len(df_geo):,}")

In [ ]:
# Create the density heatmap
fig = px.density_mapbox(
    df_geo,
    lat="latitude",
    lon="longitude",
    z="price",  # controls the intensity based on price
    radius=15,  
    center=dict(lat=54, lon=-1.5),
    zoom=5,
    mapbox_style="carto-positron",
    color_continuous_scale="YlOrRd",
    range_color=(0, 100_000_000), # Set colour range
    title="UK Property Price Density Map"
)

# Add Severn–Wash Line (Good North/South divide visual)
severn_lat, severn_lon = 51.5, -2.7
wash_lat, wash_lon = 53.0, 0.5
label_lat = (severn_lat + wash_lat) / 2
label_lon = (severn_lon + wash_lon) / 2

fig.add_trace(go.Scattermapbox(
    mode="lines",
    lat=[severn_lat, wash_lat],
    lon=[severn_lon, wash_lon],
    line=dict(color='black', width=2),
    name='Severn–Wash Line',
    showlegend=True
))

# Add label to the line
fig.add_trace(go.Scattermapbox(
    mode="text",
    lat=[label_lat],
    lon=[label_lon],
    text=["Severn–Wash Line"],
    textfont=dict(size=14, color='black'),
    showlegend=False
))

# North and South labels
fig.add_trace(go.Scattermapbox(
    mode="text",
    lat=[54],
    lon=[-1],
    text=["North"],
    textfont=dict(size=16, color='blue'),
    showlegend=False
))

fig.add_trace(go.Scattermapbox(
    mode="text",
    lat=[51],
    lon=[-1],
    text=["South"],
    textfont=dict(size=16, color='blue'),
    showlegend=False
))

# Adjust layout
fig.update_layout(
    height=900,
    width=800,
    title_x=0.5
)

fig.show()

#### Figure 4: UK Property Price Density Map
The map clearly shows a significant concentration of higher property prices in the South of England, especially in and around London and the South East. This aligns with general knowledge about the UK property market. The Severn-Wash Line effectively highlights a notable North-South divide in property prices. The South generally has much higher price density (darker reds) compared to the North (lighter yellows and oranges), although there are still pockets of higher density in the North, such as around major cities like Manchester and Leeds.

In [ ]:
# Calculate average price per postcode to reduce the number of datapoints and improve performance and readability
# Use 'first' to get a value town for each postcode.
avg_price_df = (
    df_geo.groupby('postcode')
    .agg({
        'price': 'mean',
        'latitude': 'mean',
        'longitude': 'mean',
        'town': 'first'  
    })
    .reset_index()
)

# Normalise price and generate colours
custom_price_min = 50000
custom_price_max = 2500000

# Normalise price between 0 and 1
avg_price_df['price_norm'] = (avg_price_df['price'] - custom_price_min) / (custom_price_max - custom_price_min)
avg_price_df['price_norm'] = avg_price_df['price_norm'].clip(lower=0, upper=1) # Ensure values are within [0, 1]

# Map colours: light yellow (low) to dark red (high)
def price_to_colour_yellow_red(norm):
    # Start colour (light yellow) RGB
    r_start, g_start, b_start = 255, 255, 153
    # End colour (dark red) RGB
    r_end, g_end, b_end = 139, 0, 0

    # Interpolate each colour component
    r = int(r_start + (r_end - r_start) * norm)
    g = int(g_start + (g_end - g_start) * norm)
    b = int(b_start + (b_end - b_start) * norm)

    return [r, g, b, 180] 

avg_price_df['fill_colour'] = avg_price_df['price_norm'].apply(price_to_colour_yellow_red)

# Create pydeck ColumnLayer 
layer = pdk.Layer(
    'ColumnLayer',
    data=avg_price_df,
    get_position='[longitude, latitude]',
    get_elevation='price',
    elevation_scale=0.01,
    radius=800,
    get_fill_color='fill_colour',
    pickable=True,
    auto_highlight=True
)

# Define viewstate
view_state = pdk.ViewState(
    latitude=53.0,
    longitude=-1.5,
    zoom=5,
    pitch=50
)

# Build and show the map
r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip={"text": "Town: {town}\nPostcode: {postcode}\nAvg Price: £{price}"}
)

r.show()

#### Figure 5: 3D Property Price Spikes Across the UK (Interactive)
The map clearly shows notable concentrations of high value property sales in and around major urban centres, particularly Greater London and the South East of England. Other dense clusters are visible around large cities like Manchester, Birmingham, Leeds and Bristol. This aligns with population density and economic activity. Some coastal areas and towns show notable clusters of higher value sales, which could be indicative of popular residential or holiday home markets.

<a id="22-distributions--group-comparisons-mid-level"></a>
### 2.2 Distributions & Group Comparisons (Mid-Level)

In [ ]:
# Count of number of transactions per quarter to understand market activity
# Ensure date is datetime and create year_quarter
df_residential['date'] = pd.to_datetime(df_residential['date'])
df_residential['year_quarter'] = df_residential['date'].dt.to_period('Q').astype(str)
df_residential['year'] = df_residential['date'].dt.year

In [ ]:
# Boxplots to show price distribution by Region
plt.figure(figsize=(12, 8))
sns.boxplot(data=df_residential, x='region', y='price')
plt.title('Price Distribution by Region', fontsize=16)
plt.xlabel('region', fontsize=12)
plt.ylabel('Price (£)', fontsize=12)
plt.xticks(rotation=45)  # improves readability
plt.tight_layout()
plt.show()

#### Figure 6: Distribution of Property Prices by Region (Boxplot)
London stands out with the highest median property price and the widest spread of prices, indicating the greatest variability. It also has a very high number of outliers at significantly elevated prices, with some properties exceeding £8 million. This reinforces the previous map's observation about London being a major price hotspot.
Following London, regions like the South East, East of England, and South West also show higher median prices and a greater number of high-value outliers compared to the northern and midland regions.
Almost all regions show outliers, indicating that even in generally lower-priced areas, there are individual properties that command significantly higher values than the majority. However, the magnitude and frequency of these high-value outliers are much greater in the southern regions, especially London.

In [ ]:
# 1. Create the mapping dictionary
type_mapping = {
    'D': 'Detached',
    'S': 'Semi-Detached',
    'T': 'Terraced',
    'F': 'Flats/Maisonettes',
    'O': 'Other'
}

# 2. Create the new column 'property_type_label'
df_residential['property_type_label'] = df_residential['property_type'].map(type_mapping)

# Boxplots to show price distribution by property types and new build status
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_residential, x='property_type_label', y='price', hue='new_build', palette='Set2')

plt.title("Price Distribution by Property Type and New Build Status", fontsize=16)
plt.xlabel("Property Type", fontsize=12)
plt.ylabel("Price (£)", fontsize=12)
plt.legend(title='New Build', loc='upper right')
plt.tight_layout()
plt.show()

#### Figure 7: Distribution of Property Prices by Property Type (Boxplot)
Detached properties are generally priced higher, as indicated by their higher median and upper quartile values compared to other property types, for both new and non-new builds. They also have the widest spread of prices and the highest number of high-value outliers, with some detached non-new build properties exceeding £8 million.
Flats generally have the lowest median prices and the tightest distribution, indicating they are typically the most affordable property type. They also have fewer and less extreme high-value outliers compared to detached homes.

In [ ]:
# Boxplots to show price distribution by tenure types
plt.figure(figsize=(12, 6))
sns.boxplot(x='freehold', y='price', data=df_residential)
plt.title('Price distribution by Tenure. Freehold Vs. Leasehold', fontsize=16)
plt.xlabel("Tenure", fontsize=12)
plt.ylabel("Price (£)", fontsize=12)
plt.show()

#### Figure 8: Property Prices by Tenure (Boxplot)
Freehold properties show a higher median price compared to leasehold properties. The box for freehold properties is larger, indicating a wider interquartile range and greater variability in prices. The whiskers also extend further, and there's a substantial number of high-value outliers for freehold properties, with some reaching close to £9 million. 

In [ ]:
# Boxplots to show price distribution by property and tenure types
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_residential, x='property_type_label', y='price', hue='freehold', palette='Set2')

# Formatting
plt.title("Price Distribution by Property Type and Tenure", fontsize=16)
plt.xlabel("Property Type", fontsize=12)
plt.ylabel("Price (£)", fontsize=12)
plt.legend(title='Tenure', loc='upper right') # Changed legend title to 'Tenure'
plt.tight_layout()
plt.show()

#### Figure 9: Property Prices by Tenure and Property Type (Boxplot)
For all property types, freehold properties consistently have higher median prices and a significantly wider range of values, including a greater number of high-value outliers. This is more pronounced for Detached properties, where freehold outliers reach the highest prices on the chart.
The extreme high-value outliers are almost always freehold properties across all types. This suggests that the very top end of the market, regardless of property type, is dominated by freehold tenure.
This chart provides interesting insights into how the legal ownership interacts with property type to influence price distribution, highlighting the premium associated with freehold ownership, especially for houses.

In [ ]:
# Histogram to show ditribution of house prices by property type
plt.figure(figsize=(12, 6))
sns.histplot(data=df_residential, x='price', hue='property_type_label', multiple='stack', bins=200) # Split the bars by property type
plt.title('Price Distribution by Property Type', fontsize=16)
plt.xlabel('Price')
plt.ylabel('Count')
plt.xlim(0, 1_000_000)  # x limit adjusted to focus on the majority of properties
plt.show()

#### Figure 10: Price Distribution by Property Type (Stacked Histogram)
This stacked histogram reveals a strong left-skew, with the bulk of transactions occurring under £300,000 and a sharp decline in counts for properties exceeding £500,000.
Flats have a lot of sales in the very lowest price bins, meaning they are the most affordable property type and entry point into the market.
Terraced and Semi-detached properties show high counts in the mid-price ranges, peaking roughly between £150,000 and £300,000. 
Detached properties have values across a wider price spectrum, becoming the dominant property type in the higher price bins.   
This stacked histogram visually confirms the price hierarchy observed in previous plots Flats < Terraced < Semi-detached < Detached, by showing which property type contributes most to sales at various price levels.

In [ ]:
# Log-Transformed Price Distributions
df_residential['log_price'] = np.log(df_residential['price'])

plt.figure(figsize=(12, 8))
sns.histplot(data=df_residential, x='log_price', hue='property_type_label', kde=True)
plt.title('Log-Transformed Price Distributions', fontsize=16)
plt.xlabel('Log(Price)')
plt.ylabel('Count')
plt.show()

#### Figure 11: Histogram of Log-Transformed Property Prices
The log transformation effectively separates the price distributions of the different property types more clearly than the raw price plots.   
**Flats:** Show a distribution concentrated at the lowest log-price values, indicating their affordability.   
**Terraced and Semi-detached:** Their distributions largely overlap and are positioned in the mid-range of log-prices, confirming their similar market segments.   
**Detached:** This distribution is clearly shifted to the highest log-price values, further reinforcing that detached properties generally sell for significantly more than other types.

<a id="23-time-series-analysis-fine-detail"></a>
### 2.3 Time Series Analysis (Fine Detail)

In [ ]:
# Analyse Price Trends Over Time by Region
# Create year-quarter column
df_residential['year_quarter'] = df_residential['date'].dt.to_period('Q')

# Group by region and quarter
quarterly_region_prices = (
    df_residential.groupby(['region', 'year_quarter'], as_index=False)['price']  
    .mean()
)

# Convert period to string for plotting
quarterly_region_prices['year_quarter'] = quarterly_region_prices['year_quarter'].astype(str)

# Plot
plt.figure(figsize=(14, 8))
sns.lineplot(
    data=quarterly_region_prices,  
    x='year_quarter',
    y='price',
    hue='region',
    palette='tab10',
    marker='o'  # Added markers for better visibility of data points
)

# Formatting
plt.title('Average Quarterly Property Prices by Region', fontsize=16)
plt.xlabel('Year-Quarter', fontsize=12)
plt.ylabel('Average Price (£)', fontsize=12)
plt.xticks(rotation=45, ha='right')  
plt.legend(title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Show plot
plt.show()

#### Figure 12: Average Property Prices Over Time by Region
London consistently has the highest average property prices throughout the entire period, often significantly higher than other regions. It also shows high volatility with sharp peaks and troughs.
The chart shows a tiered price structure across regions. London is at the top, followed by the South East, East of England, and South West, which form a second tier of higher-priced regions. The remaining regions (East Midlands, North East, North West, Wales, West Midlands, Yorkshire and the Humber) generally occupy the lower tiers, with average prices well below the southern regions. This reinforces the North-South divide observed in the earlier density map.
For most regions, there was a general upward trend in average property prices from 2015 until around late 2021 or early 2022. This suggests a period of sustained growth in the UK property market.

In [ ]:
# Count of number of transactions per quarter to understand market activity
# Ensure date is datetime and create year_quarter
df_residential['date'] = pd.to_datetime(df_residential['date'])
df_residential['year_quarter'] = df_residential['date'].dt.to_period('Q').astype(str)

# Group by quarter and count transactions
transaction_volume = df_residential.groupby('year_quarter').size().reset_index(name='transactions')

# Plot
plt.figure(figsize=(12, 8))
sns.lineplot(data=transaction_volume, x='year_quarter', y='transactions', linewidth=1.5)

# Highlight 2020Q1 to 2020Q3 (COVID-19)
plt.axvspan('2020Q1', '2020Q3', color='red', alpha=0.1)
plt.annotate('COVID-19 Impact',
             xy=('2020Q2', transaction_volume[transaction_volume['year_quarter'] == '2020Q2']['transactions'].values[0]),
             xytext=('2019Q4', transaction_volume['transactions'].max() * 0.95),
             fontsize=12,
             backgroundcolor='white')

# Highlight 2021Q2 to 2021Q4 (End of Stamp Duty Holiday)
plt.axvspan('2021Q2', '2021Q4', color='blue', alpha=0.1)
plt.annotate('End of Stamp Duty Holiday',
             xy=('2021Q3', transaction_volume[transaction_volume['year_quarter'] == '2021Q3']['transactions'].values[0]),
             xytext=('2020Q4', transaction_volume['transactions'].max() * 0.85),
             fontsize=12,
             backgroundcolor='white')

# Highlight 2022Q3 to 2023Q1 (Mini-Budget)
plt.axvspan('2022Q3', '2023Q1', color='orange', alpha=0.1)
plt.annotate('Mini-Budget Impact (Liz Truss)',
             xy=('2022Q4', transaction_volume[transaction_volume['year_quarter'] == '2022Q4']['transactions'].values[0]),
             xytext=('2022Q1', transaction_volume['transactions'].max() * 0.88),
             fontsize=12,
             backgroundcolor='white')

# Formatting
plt.title('Number of Sales per Quarter', fontsize=16)
plt.xlabel('Year-Quarter', fontsize=12)
plt.ylabel('Number of Transactions', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

### Figure 13: Number of Property Sales Per Year
Before the COVID-19 pandemic, the number of sales per quarter generally fluctuated within a similar range and showing a degree of seasonality.   
**COVID-19:** Caused a sharp fall in sales, down to ~1,250 sales.   
**Stamp Duty Holiday:** Triggered a massive surge, peaking at 3,500+ sales.   
**Mini-Budget Fallout:** Another sharp drop post-2022, reflecting loss in market confidence which has continues throughout 2023 and 2024.

In [ ]:
# Group by property type and quarter, compute mean price
quarterly_property_prices = (
    df_residential.groupby(['property_type_label', 'year_quarter'], as_index=False)['price']
    .mean()
)

# Convert year_quarter to string
quarterly_property_prices = quarterly_property_prices.sort_values('year_quarter')
quarterly_property_prices['year_quarter'] = quarterly_property_prices['year_quarter'].astype(str)

# Plot
plt.figure(figsize=(14, 8))
sns.lineplot(
    data=quarterly_property_prices,
    x='year_quarter',
    y='price',
    hue='property_type_label',
    palette='tab10',
    marker='o',
    ci=None
)

# Formatting
plt.title('Average Quarterly Property Prices by Property Type', fontsize=16)
plt.xlabel('Year-Quarter', fontsize=12)
plt.ylabel('Average Price (£)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Property Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Show plot
plt.show()

#### Figure 14: Average Property Prices Over Time by Property Type
This chart strongly reiterates the established price hierarchy among the four property types. Detached properties consistently maintain the highest average prices, followed by Semi-detached, then Terraced, and Flats remaining the most affordable.
Detached properties, despite being the most expensive, show high volatility. However, they maintain their high price indicating sustained demand for larger homes.
A general, steady upward trend is visible across all property types.

In [ ]:
# Define price bins
bins = np.linspace(0, 1_000_000, 200)

# Generate a colour palette for property types
property_types = df_residential['property_type_label'].dropna().unique()
palette = sns.color_palette("tab10", len(property_types))
custom_palette = dict(zip(property_types, palette))

# Create the plot
plt.figure(figsize=(12, 6))

for ptype, color in custom_palette.items():
    subset = df_residential[df_residential['property_type_label'] == ptype]
    counts, edges = np.histogram(subset['price'], bins=bins)
    centers = 0.5 * (edges[1:] + edges[:-1])  # bin centers

    plt.plot(centers, counts, label=ptype, color=color, linewidth=2)

# Formatting
plt.title('Price Distribution by Property Type (Count Line Plot)', fontsize=16)
plt.xlabel('Price (£)')
plt.ylabel('Number of Sales')
plt.xlim(0, 1_000_000)
plt.legend(title='Property Type')
plt.tight_layout()
plt.show()

### Figure 15: Price Distribution by Property Type
Each property type shows a distinct price clustering, with evident peaks at psychologically significant price points (£200k, £300k, etc.)   
**Flats** have the lowest price distribution, peaking significantly around £100,000-£150,000. Their sales numbers drop off sharply after £200,000, indicating they are mostly in the lower to mid-price segments.   
**Terraced** show a distribution slightly higher than flats, with a peak around £150,000-£200,000. Their curve extends further into the mid-price range than flats.   
**Semi-Detached** are in a mid-to-higher price range, with their peak sales occurring around £200,000-£250,000. The distribution stretches significantly towards higher prices compared to flats and terraced properties.   
**Detached** have the highest price distribution. While they have a peak around £250,000-£300,000, their curve is much broader and extends far further into the higher price ranges (up to £1,000,000 and beyond) with considerable sales volume, even at these elevated prices. This confirms that detached properties are the most expensive property type.

In [ ]:
# Create density distribution chart
plt.figure(figsize=(12, 6))

# Loop through property types and plot KDE line for each
for ptype, color in custom_palette.items():
    subset = df_residential[df_residential['property_type_label'] == ptype]
    sns.kdeplot(
        data=subset,
        x='price',
        bw_adjust=1,       # bandwidth adjustment
        common_norm=False,
        fill=False,
        linewidth=2,
        color=color,
        label=ptype,
        clip=(0, 1_000_000)  # limit x range
    )

# Formatting
plt.title('Price Distribution by Property Type (Line Plot)', fontsize=16)
plt.xlabel('Price')
plt.ylabel('Density')
plt.xlim(0, 1_000_000)
plt.legend(title='Property Type')
plt.tight_layout()
plt.show()

### Figure 16: Price Distribution by Property Type (Density Line Plot)
Unlike the previous line plot, this density version smooths the volume variation giving a good visualiation revealing the underlying pricing tendencies of each property type. 
Each property type has a distinct segment of the price spectrum, with Detached properties again having the most high-value distribution.

## Summary of Exploratory Data Analysis

I have completed a comprehensive exploratory data analysis of the dataset. This phase has involved:

* **Geographical Analysis:** Geocoding postcodes and visualising property price distribution using choropleth maps and 3D visualisations.
* **Distribution and Group Comparisons:** A detailed look at price distributions across regions, property types, new build status, and tenure using various plots including boxplots, histograms, and density plots.
* **Time Series Analysis:** An in-depth look at quarterly prices to spot trends and highlight key insights in the data.

With these initial insights, the data is now prepared for a deeper, more targeted analysis to address the key project questions.

<a id="3-statistical-analysis-and-visualisations"></a>
## 3. Statistical Analysis and Visualisations
This section focuses on answering our key research questions using statistical analysis and predictive modeling. I will explore historical trends and regional variations to gain a better understanding of the UK housing market.

<a id="31-pricing-forecast"></a>
### 3.1 Pricing Forecast
#### Focus Region: North West
As I work for a housing association in the North West I will focus on this region to simulate a real-world use case.

In [ ]:
# Filter just North West region residential data
nw_df = df_residential[df_residential['region'] == 'North West'].copy()

# Parse the date column
nw_df['date'] = pd.to_datetime(nw_df['date'])

In [ ]:
# Create year-month column and aggregate
nw_df['year_month'] = nw_df['date'].dt.to_period('M')
nw_monthly = nw_df.groupby('year_month')['price'].mean().reset_index()

# Convert back to datetime for modeling
nw_monthly['year_month'] = nw_monthly['year_month'].dt.to_timestamp()
nw_monthly = nw_monthly.set_index('year_month')

# Apply the log transformation to the price
nw_monthly['log_price'] = np.log(nw_monthly['price'])

In [ ]:
# Plot the Time Series for North West
plt.figure(figsize=(14, 6))
sns.lineplot(data=nw_monthly, x=nw_monthly.index, y='price')
plt.title("Monthly House Prices in the North West", fontsize=16)
plt.ylabel("Average Price (£)")
plt.xlabel("Date")
plt.show()

### Figure 17: Monthly Average House Prices in the North West
There is a clear upward trend with some periods of stagnation. This time series shows patterns that are suitable for forecasting models and I will use a time-series model to predict the next 3 years' prices.

### Time Series Forecasting with SARIMA

To predict future property prices, I will use a Seasonal AutoRegressive Integrated Moving Average (SARIMA) model. This is a powerful time series model that can account for trend, seasonality, and other patterns in the data.

The general form of a SARIMA model is: $SARIMA(p,d,q)(P,D,Q)_s$

* $p,d,q$: represent the non-seasonal components.
* $P,D,Q$: represent the seasonal components.
* $s$: represents the seasonality

In [ ]:
# Decompose the Time Series
log_prices = np.log(nw_monthly['price'])
decomposition = seasonal_decompose(log_prices, model='additive', period=12)
decomposition.plot()
plt.suptitle('Time Series Decomposition (Log-Transformed) - North West', fontsize=16)
plt.tight_layout()
plt.show()

### Figure 18: Time Series Decomposition of North West House Prices
The **Trend** plot confirms the upward trend for house prices. The **Seasonal** plot hows a repeating annual pattern indicating prices tend to fluctuate in a predictable annual cycle. The **Residuals** plot shows the remaining noise after the trend and seasnoality have been removed. The pattern here supports the use of a **SARIMA** model. 

In [ ]:
# Apply seasonal differencing to log-transformed data
log_seasonal_diff = log_prices.diff(periods=12).dropna()

# Augmented Dickey-Fuller Test on seasonally differenced log data
adf_result_seasonal = adfuller(log_seasonal_diff)

print("ADF Test Results - Seasonally Differenced Log Series:")
print(pd.Series(adf_result_seasonal[0:4], index=['Test Statistic', 'p-value', '#Lags Used', 'Number of Observations Used']))
print(f"Critical Values: {adf_result_seasonal[4]}")

### Table 5: Augmented Dickey-Fuller (ADF)
To check for **seasonal stationarity**, I performed an ADF test on the 12-month (seasonally) differenced data.

The ADF test statistic of -5.60 is well below all the critical values, and the p-value of 0.000001 is below the 0.05 threshold. This provides strong evidence to reject the null hypothesis of a unit root, indicating that the seasonally differenced log-transformed series is **stationary**.

This confirms the suitability of a **SARIMA model**

In [ ]:
# First difference of log-transformed prices
log_diff = log_prices.diff().dropna()

# Plot first-order differenced log series
plt.figure(figsize=(12, 5))
plt.plot(log_diff)
plt.title("First Order Differenced Log Series - North West", fontsize=16)
plt.xlabel("Date")
plt.ylabel("Differenced log(Price)")
plt.grid(True)
plt.show()

### Figure 19: First Order Differenced Series
The plot fluctuates around a relatively constant mean near zero, with a more stable variance compared to the original series. This visual pattern suggests that the first-order differencing of the log-transformed data has effectively removed the longterm trend, making the series more likely to be stationary.

In [ ]:
# ACF and PACF plots
plt.figure(figsize=(14, 5))
plt.subplot(121)
plot_acf(log_diff, lags=40, ax=plt.gca(), title='ACF - First-Differenced Log Series (North West)')
plt.subplot(122)
plot_pacf(log_diff, lags=40, ax=plt.gca(), title='PACF - First-Differenced Log Series (North West)')
plt.tight_layout()
plt.show()

### Figure 20: ACF and PACF Plots for First-Order Differenced Series
The ACF plot (left) shows a significant spike at lag 1 then it drops off into the confidence interval. This suggests a moving average component.

The PACF plot (right) also shows a significant spike at lag 1 and then quickly drops off. There are other significant spikes at later lags, particularly a strong negative spike at lag 18. This suggests a possibly more complex long-term pattern that isn't a simple 12-month cycle.

Based on this information I will try the following values:   
- $d = 1$: As confirmed by the **Augmented Dickey-Fuller test** (ADF), differencing was required to achieve stationarity.
- $p = 3$: From the PACF plot, lag 4 is the first within the confidence interval, so p = 3 but 5 is also a possible canditate and with a strong spike at 1 I will try a few different versions.
- $q = 1$: From the ACF plot, lag 2 is the first within the interval, so q = 1.

In [ ]:
# ACF & PACF on seasonally differenced data
# Define seasonally differenced log data
nw_seasonal_diff = log_prices.diff(12).dropna()
plt.figure(figsize=(14, 6))
plt.subplot(121)
plot_acf(nw_seasonal_diff, lags=36, ax=plt.gca(), title='ACF - Seasonally Differenced')
plt.subplot(122)
plot_pacf(nw_seasonal_diff, lags=36, ax=plt.gca(), title='PACF - Seasonally Differenced')
plt.tight_layout()
plt.show()

### Figure 21: ACF and PACF Plots for the Seasonally Differenced Data

Based on this information I will try the following values:$P$ = 0, $D$ = 1, $Q$ = 1. But I will also try a few other variations.

<a id="32-modelling-decisions-and-sarima-forecasting"></a>
### 3.2 Modelling Decisions and SARIMA Forecasting

Having confirmed the presence of both **trend** and **seasonality** in the data, I chose to use a **Seasonal ARIMA (SARIMA)** model. The model accounts for both short-term dynamics and annual seasonality which is a logical choice given the cyclical nature of housing prices.

The main objective is to find the model that provides the best fit to the data while avoiding overfitting. To achieve this, I compare multiple models using:

- Variations in **non-seasonal parameters**: $(p,d,q)$
- Variations in **seasonal parameters**: $(P,D,Q)$
- Model selection criteria: **Akaike Information Criterion (AIC)** 

The model with the lowest AIC will be selected as the best performer.

In [ ]:
# Fit SARIMA Model

# Prepare the log-transformed time series for the North West region
nw_df_ARIMA = df_residential[df_residential['region'] == 'North West'].copy()

# Set datetime index and resample to monthly frequency
nw_df_ARIMA = nw_df_ARIMA.set_index('date')['price'].resample('MS').mean()
nw_df_ARIMA = pd.DataFrame(nw_df_ARIMA)  # Ensure it's a DataFrame
nw_df_ARIMA['log_price'] = np.log(nw_df_ARIMA['price'])

# Define y as the,log-transformed series
y = nw_df_ARIMA['log_price'].dropna()

# Set the frequency of the data
nw_monthly = nw_monthly.asfreq('MS')

# Candidate models for comparison
models_to_compare = {
    'SARIMA(3,1,2)(0,1,1,12)': ((3, 1, 2), (0, 1, 1, 12)), 
    'SARIMA(3,1,1)(1,1,1,12)': ((3, 1, 1), (1, 1, 1, 12)),
    'SARIMA(3,1,2)(1,1,1,12)': ((3, 1, 2), (1, 1, 1, 12)),
    'SARIMA(3,1,2)(1,1,0,12)': ((3, 1, 2), (1, 1, 0, 12)),
    'SARIMA(2,1,2)(1,1,1,12)': ((2, 1, 2), (1, 1, 1, 12)),
    'SARIMA(1,1,1)(1,1,1,12)': ((1, 1, 1), (1, 1, 1, 12)),
    'SARIMA(1,1,1)(0,1,1,12)': ((1, 1, 1), (0, 1, 1, 12)),  
    'SARIMA(5,1,1)(1,1,1,12)': ((5, 1, 1), (1, 1, 1, 12)),
    'SARIMA(3,1,3)(0,1,1,12)': ((3, 1, 3), (0, 1, 1, 12)), 
    'SARIMA(4,1,2)(0,1,1,12)': ((4, 1, 2), (0, 1, 1, 12)),
    'SARIMA(3,1,1)(0,1,2,12)': ((3, 1, 1), (0, 1, 1, 12)),
}

# Dictionary to store results
aic_results = {}

print("Comparing SARIMA models by AIC and BIC...")

for name, params in models_to_compare.items():
    order, seasonal_order = params
    try:
        model = SARIMAX(y,
                        order=order,
                        seasonal_order=seasonal_order,
                        enforce_invertibility=False,
                        enforce_stationarity=False)       
        results = model.fit(disp=False, maxiter=500) # maxiter helps with convergance issue
        aic_results[name] = {'AIC': results.aic, 'BIC': results.bic}
        print(f"Model: {name} | AIC: {results.aic:.2f} | BIC: {results.bic:.2f}")
    except Exception as e:
        print(f"Failed to fit {name}: {e}")

In [ ]:
# Identify the best model based on the lowest AIC
best_model_name = min(aic_results, key=lambda k: aic_results[k]['AIC'])
best_model_aic = aic_results[best_model_name]['AIC']
best_model_params = models_to_compare[best_model_name]

print(f"\nThe best-performing model based on the lowest AIC is: {best_model_name}")
print(f"Its AIC is: {best_model_aic:.2f}")

In [ ]:
# Fit final SARIMA model
sarima_model = SARIMAX(
    y,  # log-transformed data
    order=(1, 1, 1),
    seasonal_order=(0, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_results = sarima_model.fit()
print(sarima_results.summary())

### Table 6: SARIMA Model Summary
This displays displays the summary of the final fitted **SARIMA(1,1,1)(0,1,1,12)** model. This model had the lowest AIC, indicating the best trade-off between fit and complexity.
The SARIMA model fits the data well overall, shown by the good residual diagnostics.   
The **ma.L1** term is highly significant, showing a strong moving average effect, while the ar.L1 and seasonal MA.S.L12 terms are not statistically significant.   
Despite some non-significant terms, the model still effectively captures the underlying trend and seasonality, showing that not all parameters need to be significant for good forecasting performance.

In [ ]:
# Forecasting next 3 years
forecast_steps = 36
forecast = sarima_results.get_forecast(steps=forecast_steps)
forecast_mean = forecast.predicted_mean
conf_int = forecast.conf_int()

# Convert forecast back to original price scale
forecast_mean = np.exp(forecast_mean)
conf_int = np.exp(conf_int)

# Generate future dates
forecast_index = pd.date_range(
    start=y.index[-1] + pd.DateOffset(months=1),  
    periods=forecast_steps,
    freq='MS'
)

# Convert historical data back to price scale
historical_prices = np.exp(y)

# Plot original + forecast
plt.figure(figsize=(14, 6))
plt.plot(historical_prices.index, historical_prices.values, label='Historical Data')
plt.plot(forecast_index, forecast_mean, label='Forecast', linestyle='--', color='darkorange')
plt.fill_between(forecast_index, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                 color='orange', alpha=0.2, label='95% Confidence Interval')

# Formatting
plt.title("SARIMA Forecast - North West Region (Next 3 Years)", fontsize=16)
plt.xlabel("Date")
plt.ylabel("Average House Price (£)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Figure 22: SARIMA Forecast for North West House Prices (Next 3 Years)
This plot shows the model’s forecast for average house prices over the next **3 years**, including a 95% confidence interval. The forecast suggests a continued upward trend in prices for the North West.

<a id="33-comparing-arima-and-sarima-forecasting-models"></a>
### 3.3 Comparing ARIMA and SARIMA Forecasting Models

To assess predictive accuracy in a stable market period, I compared an **ARIMA** model and a **SARIMA** model  

In [ ]:
# Fit the ARIMA Model (Non-Seasonal)
arima_model = ARIMA(y, order=(1, 1, 1)) # Use the same values as SARIMA model
arima_fit = arima_model.fit()
print("ARIMA Model Summary:")
print(arima_fit.summary())
print(f"ARIMA Model AIC: {arima_fit.aic:.2f}")

### Table 7: ARIMA Model Summary 
The ARIMA(1,1,1) model provides a strong fit to the log-transformed house price data with a low AIC of -219.72.

In [ ]:
# Model Comparison
print("Final Model Comparison:")
print(f"ARIMA(1,1,1) AIC: {arima_fit.aic:.2f}")
print(f"SARIMA(1,1,1)(0,1,1,12) AIC: {sarima_results.aic:.2f}")

In [ ]:
# Plot ARIMA and SARIMA results
n_forecast_steps = 36

# Forecasts (log scale)
arima_forecast = arima_fit.get_forecast(steps=n_forecast_steps)
sarima_forecast = sarima_results.get_forecast(steps=n_forecast_steps) 

# Predicted means
arima_forecast_mean = arima_forecast.predicted_mean
sarima_forecast_mean = sarima_forecast.predicted_mean

# Confidence intervals
arima_ci = arima_forecast.conf_int()
sarima_ci = sarima_forecast.conf_int()

# Convert back from log scale to original price scale
historical_prices = np.exp(y)
arima_forecast_prices = np.exp(arima_forecast_mean)
sarima_forecast_prices = np.exp(sarima_forecast_mean)
arima_ci_prices = np.exp(arima_ci)
sarima_ci_prices = np.exp(sarima_ci)

# Generate forecast datetime index
forecast_index = pd.date_range(
    start=y.index[-1] + pd.DateOffset(months=1),
    periods=n_forecast_steps,
    freq='MS'
)

# Assign the forecast index to the forecasts and confidence intervals
arima_forecast_prices.index = forecast_index
sarima_forecast_prices.index = forecast_index
arima_ci_prices.index = forecast_index
sarima_ci_prices.index = forecast_index

In [ ]:
# Plot model comparisons
plt.figure(figsize=(18, 8))
plt.title('ARIMA vs. SARIMA Forecast Comparison for North West Region', fontsize=20)

# Historical actual prices
plt.plot(historical_prices.index, historical_prices.values, label='Historical Data (Actual Price)', linewidth=2)

# ARIMA forecast + CI
plt.plot(arima_forecast_prices.index, arima_forecast_prices.values, color='red', label='ARIMA(1,1,1) Forecast', linestyle='--')
plt.fill_between(arima_ci_prices.index, arima_ci_prices.iloc[:, 0], arima_ci_prices.iloc[:, 1], color='red', alpha=0.1, label='ARIMA 95% CI')

# SARIMA forecast + CI
plt.plot(sarima_forecast_prices.index, sarima_forecast_prices.values, color='darkorange', label='SARIMA(1,1,1)(0,1,1,12) Forecast', linestyle='-.')
plt.fill_between(sarima_ci_prices.index, sarima_ci_prices.iloc[:, 0], sarima_ci_prices.iloc[:, 1], color='darkorange', alpha=0.1, label='SARIMA 95% CI')

# Formatting
plt.xlabel('Date', fontsize=14)
plt.ylabel('Average Price (£)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Figure 23: ARIMA vs. SARIMA forecast comparison
The ARIMA model shows a flat, constant forecast for future prices (dashed red line) where the SARIMA model shows a more dynamic forecast with seasonality.  While the ARIMA model has a lower AIC, indicating a statistically better fit to the data, the SARIMA model, despite its higher AIC value, looks to be a more suitable forecasting tool for this dataset.

### SARIMA Model Diagnostics
This figure provides a diagnostic check of the residuals from our fitted SARIMA model. Diagnostic plots are useful for checking if model assumptions have been met.

In [ ]:
# SARIMA Residuals plot
sarima_results.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()

### Figure 24: SARIMA model diagnostics

- **Standardised Residuals:** No obvious trends, residuals are centered around zero.
- **Histogram & KDE:** Approximates a normal distribution, supports the assumption of normal errors.
- **Normal Q-Q Plot:** Points lie close to the red line, again supporting normality.
- **Correlogram (ACF of residuals):** No significant autocorrelation left in residuals, confirms a good model fit.

These diagnostics suggest the model’s residuals behave like white noise, confirming that it captures the time series structure effectively.

In [ ]:
# Define new start date
new_start = pd.to_datetime('2016-07-01')

# Get sample predictions starting from new_start
fitted = sarima_results.get_prediction(start=new_start, end=y.index[-1], dynamic=False)

# Convert fitted values and actual values back to original price scale
fitted_mean = np.exp(fitted.predicted_mean)
actual_prices = np.exp(y)

# Slice actual prices to match the new fitted period
actual_trimmed = actual_prices[actual_prices.index >= new_start]

# Plot Actual vs Fitted
plt.figure(figsize=(14, 6))
plt.plot(actual_trimmed.index, actual_trimmed.values, label='Actual', linewidth=2)
plt.plot(fitted_mean.index, fitted_mean.values, label='Fitted (SARIMA)', linestyle='--', color='darkorange')

# Formatting
plt.title("SARIMA Model: Actual vs Fitted - North West", fontsize=16)
plt.xlabel("Date")
plt.ylabel("Average House Price (£)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Figure 25: SARIMA Model: Actual vs Fitted 
This plot compares the model's fitted values with the actual observed prices.
An anomalous spike in early 2016 was observed, so the plot was adjusted to begin after this point to provide a clearer view of the model's fit.   
The model captures the overall upward trend and seasonal fluctuations well.   
While the model doesn’t capture every fluctuation, it shows a strong alignment with the broader patterns, supporting its reliability for forecasting.   
This visual fit supports the forecasting reliability of the SARIMA model.

<a id="34-summary-of-time-series-analysis"></a>
### 3.4 Summary of Time Series Analysis

This concludes the time series analysis and forecasting for the North West region.

### Key Findings:

- **Trend & Seasonality:** Strong upward trend and annual cycles in the data.
- **Best Model:** The SARIMA(1,1,1)(0,1,1,12) model was selected for its ability to capture both trend and seasonal patterns, despite the ARIMA(1,1,1) model having a slightly lower AIC.
- **Forecasting:** The model projects a continued increase in house prices over the next three years, altough forecase uncertainty grows over time.
- **Diagnostics:** Residuals were approximately normally distributed, showed no autocorrelation, and exhibited no clear patterns — validating the model’s assumptions.

This model now serves as a solid foundation for evaluating regional market dynamics and may be extended to compare other UK regions in future work.

<a id="35-regional-analysis---north-vs-south"></a>
### 3.5 Regional Analysis - North vs. South

Having completed a detailed time series analysis for the North West, I now broaden the scope to examine regional patterns across the UK.
This section addresses the following research questions:
**How have UK property prices changed over time?**   
**Are there notable regional differences (e.g., North vs. South)?**

By comparing trends across multiple regions, this analysis will provide insights into geographic disparities, growth rates, and potential market imbalances within the UK housing sector.

In [ ]:
# Analyse Price Trends Over Time for all properties

quarterly_prices = df_residential.groupby('year_quarter')['price'].mean().reset_index()
quarterly_prices['year_quarter'] = quarterly_prices['year_quarter'].astype(str)
yearly_prices = df_residential.groupby('year')['price'].mean().reset_index()

# Create the plot
plt.figure(figsize=(12, 8))
sns.lineplot(data=quarterly_prices, x='year_quarter', y='price')

# Highlight 2020Q1 to 2020Q3 for COVID-19
plt.axvspan('2020Q1', '2020Q3', color='red', alpha=0.1)

# Annotate the dip at 2020Q2
plt.annotate('COVID-19 Impact',
             xy=('2020Q2', quarterly_prices[quarterly_prices['year_quarter'] == '2020Q2']['price'].values[0]),
             xytext=('2019Q4', quarterly_prices['price'].max() * 0.95),
             fontsize=12,
             backgroundcolor='white')

# Highlight 2021Q2 to 2021Q4 for end of stamp duty holiday
plt.axvspan('2021Q2', '2021Q4', color='blue', alpha=0.1)

# Annotate stamp duty holiday end
plt.annotate('End of Stamp Duty Holiday',
             xy=('2021Q3', quarterly_prices[quarterly_prices['year_quarter'] == '2021Q3']['price'].values[0]),
             xytext=('2020Q4', quarterly_prices['price'].max() * 0.87),
             fontsize=12,
             backgroundcolor='white')

# Highlight 2022Q3 to 2023Q1 for mini-budget
plt.axvspan('2022Q3', '2023Q1', color='orange', alpha=0.1)

# Annotate the mini-budget impact
plt.annotate('Mini-Budget Impact (Liz Truss)',
             xy=('2022Q4', quarterly_prices[quarterly_prices['year_quarter'] == '2022Q4']['price'].values[0]),
             xytext=('2022Q1', quarterly_prices['price'].max() * 0.90),
             fontsize=12,
             backgroundcolor='white')

# Formatting
plt.title('Average Quarterly Property Prices', fontsize=16)
plt.xlabel('Year-Quarter', fontsize=12)
plt.ylabel('Average Price (£)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Figure 26: Average Quarterly Property Prices.
This visualisation shows a strong long-term upward trend in property values and highlights the market's sensitivity to major economic and political events. 

In [ ]:
# 1. See all unique regions and how many records each has
region_counts = df_residential['region'].value_counts()
print("--- Unique Regions in Dataset ---")
print(region_counts)

In [ ]:
# 1. Define North/South mapping based on UK Regions
north_regions = ['North East', 'North West', 'Yorkshire and The Humber', 'Scotland', 'Wales', 'East Midlands', 'West Midlands']

def get_north_south(region):
    if region in north_regions:
        return 'North'
    else:
        return 'South'

# 2. Create 'north_south' column
df_residential['north_south'] = df_residential['region'].apply(get_north_south)

df_residential['year_quarter'] = df_residential['date'].dt.to_period('Q')

quarterly_ns_prices = (
    df_residential.groupby(['north_south', 'year_quarter'], as_index=False)['price']
    .mean()
)

# Convert period to string for plotting
quarterly_ns_prices['year_quarter'] = quarterly_ns_prices['year_quarter'].astype(str)

# --- Plotting Code ---
plt.figure(figsize=(14, 8))
ns_groups = quarterly_ns_prices['north_south'].unique()

# Define a palette manually if 'palette' isn't defined elsewhere
colors = sns.color_palette("Set1", len(ns_groups))
color_dict = dict(zip(ns_groups, colors))

for group in ns_groups:
    group_data = quarterly_ns_prices[quarterly_ns_prices['north_south'] == group]
    sns.lineplot(
        data=group_data,
        x='year_quarter',
        y='price',
        label=group,
        color=color_dict[group],
        marker='o'
    )

plt.title('Average Quarterly Property Prices by North/South', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.show()

### Figure 27: Average Quarterly Property Prices North vs. South
This chart shows the massive price differences between the North and South regions of the UK.
Both lines show the same pattern and an upward trend but it is obvious there is a huge difference in house prices between the North and South regions.

In [ ]:
# Create the north_south column in df_geo 
north_regions = ['North East', 'North West', 'Yorkshire and The Humber', 'Scotland', 'Wales', 'East Midlands', 'West Midlands'] 

df_geo['north_south'] = df_geo['region'].apply(lambda x: 'North' if x in north_regions else 'South')

# Violin Plot
plt.figure(figsize=(10, 6))
sns.violinplot(data=df_geo, x="north_south", y="price", inner="quartile", palette="pastel")
# ... rest of your code

# Violin plot of prices for North and South
plt.figure(figsize=(10, 6))

# Create the violin plot
sns.violinplot(
    data=df_geo,
    x="north_south",
    y="price",
    scale="width",
    inner="quartile",
    palette="pastel"
)

# Formatting
plt.title("Distribution of Property Prices in North vs South", fontsize=14)
plt.xlabel("Region", fontsize=12)
plt.ylabel("Price (£)", fontsize=12)
plt.yscale('log')  # log scale for better visualisation of skewed date

plt.tight_layout()
plt.show()

### Figure 28: Violin Plot for North and South Property Prices
This shows an overall price distribution for both North and South regions This plot clearly shows that property prices in the South are much higher than the North. The South's violin is wider at much higher values indicating a greater range of property prices and a larger number of very high value properties. I used a log scale to better display the data and include the outliers whilst giving a good distribution visualisation.

<a id="36-regional-price-changes"></a>
### 3.6 Regional Price Changes

### Data Preparation for Regional Price Visualisation

In [ ]:
# Create year column 
df_geo['year'] = df_geo['date'].dt.year
regional_yearly_prices = df_geo.groupby(['region', 'year'])['price'].mean().reset_index()

# Drop NaNs caused by pct_change
regional_yearly_prices.dropna(inplace=True)

In [ ]:
# Update the path to point to your Project folder
geojson_path = r"Regions.geojson"

# Load and simplify GeoJSON file
gdf = gpd.read_file(geojson_path)

# Simplify geometry
gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.01, preserve_topology=True)

# Convert back to GeoJSON format
region_geo = json.loads(gdf.to_json())

In [ ]:
# Map the Region names
name_mapping = {
    'EAST': 'East of England',
    'EAST MIDLANDS': 'East Midlands (England)',
    'LONDON': 'London',
    'NORTH EAST': 'North East (England)',
    'NORTH WEST': 'North West (England)',
    'SOUTH EAST': 'South East (England)',
    'SOUTH WEST': 'South West (England)',
    'WEST MIDLANDS': 'West Midlands (England)',
    'YORKSHIRE AND THE HUMBER': 'Yorkshire and The Humber',
}

In [ ]:
# Apply the mapping
regional_yearly_prices['region'] = regional_yearly_prices['region'].str.upper().map(name_mapping)

In [ ]:
# Create the animated choropleth map with static colour scale
fig = px.choropleth(
    regional_yearly_prices,
    geojson=region_geo,
    locations='region',
    featureidkey='properties.nuts118nm',
    color='price',
    color_continuous_scale='Viridis',
    range_color=[100000, 600000],  # Static colour scale from 100k to 600k
    animation_frame='year',
    scope='europe',  
    title='Average Yearly Property Price by Region (£)'
)

# Layout focusing on the UK and increasing map size
fig.update_geos(
    fitbounds="locations",
    visible=False,
    projection_type="mercator",
    lataxis_range=[50, 60],  # Latitude range for UK
    lonaxis_range=[-10, 5]   # Longitude range for UK
)
fig.update_layout(
    margin={"r":0, "t":50, "l":0, "b":0},
    coloraxis_colorbar_title="Price (£)",
    coloraxis_colorbar_tickprefix="£",
    width=700,  # Set width to 700 pixels 
    height=700   # Set height to 700 pixels
)

# Show the plot
fig.show()

### Figure 29: Average Yearly Property Price by Region (Interactive)
This visualisation shows the changes to property price for UK regions and highlights the widening North-South divide. By 2023, the southern regions, especially London and the South East, are becoming significantly lighter indicating higher prices. The changes over time for the whole of the UK does still show a general increasing trend as shown in the previous time series charts.

<a id="37-t-test-north-vs-south-property-prices"></a>
### 3.7 T-Test: North vs. South Property Prices
This section aims to determine if there is a statistically significant difference in individual house prices between properties located in the **North** and **South** of the UK. This regional analysis provides crucial insights for my housing association, helping us understand market disparities.

### Performing Independent Samples T-test

To formally assess if the observed difference in average prices between the North and South is statistically significant, I will use an **Independent Samples T-test**. This test is appropriate for comparing the means of two independent groups.

- **Null Hypothesis (H₀):** There is **no significant difference** in mean house prices between the North and South.    
- **Alternative Hypothesis (H₁):** There **is** a significant difference in mean house prices.  

In [ ]:
# Perform T-test
north_prices = df_residential[df_residential['north_south'] == 'North']['price'].dropna()
south_prices = df_residential[df_residential['north_south'] == 'South']['price'].dropna()

if north_prices.empty or south_prices.empty:
    print("Error")
else:
    # Perform the t-test
    t_statistic, p_value = stats.ttest_ind(north_prices, south_prices, equal_var=False)

    print(f"Number of North prices: {len(north_prices):,}")
    print(f"Number of South prices: {len(south_prices):,}")
    print(f"Mean Price (North): £{north_prices.mean():,.2f}")
    print(f"Mean Price (South): £{south_prices.mean():,.2f}")
    print(f"T-Statistic: {t_statistic:.3f}")
    print(f"P-Value: {p_value:.3e}") 

### Table 8 T-test Result:
The T-test reveals a highly significant difference in mean house prices between the North and South regions.   
With a P-value effectively zero (p < 0.001), I reject the null hypothesis.   
On average, Southern properties are substantially more expensive than Northern ones (by over £160,000), confirming a strong regional price disparity in the UK housing market.

### This concludes the analysis section of the project. 
The detailed findings are in the accompanying essay

<a id="4-list-of-tables-and-figures"></a>
## 4. List of Tables and Figures

**Table 1**: Top 5 Most Expensive Properties by Type  
**Table 2**: Bottom 5 Least Expensive Properties by Type  
**Table 3**: Suspected Low-Value Outliers by County  
**Table 4**: Price Check for Low Values  
**Table 5**: Augmented Dickey-Fuller (ADF) Test Results  
**Table 6**: SARIMA Model Summary  
**Table 7**: ARIMA Model Summary  

**Figure 1**: Overall Distribution of UK Property Prices  
**Figure 2**: Price Distribution by Property Type  
**Figure 3**: Boxplots by Individual Property Type  
**Figure 4**: UK Property Price Density Map
**Figure 5**: 3D Property Price Spikes Across the UK **(Interactive)**  
**Figure 6**: Distribution of Property Prices by Region (Boxplot)  
**Figure 7**: Distribution of Property Prices by Property Type (Boxplot)  
**Figure 8**: Property Prices by Tenure (Boxplot)  
**Figure 9**: Property Prices by Tenure and Property Type (Boxplot)  
**Figure 10**: Price Distribution by Property Type (Stacked Histogram)  
**Figure 11**: Histogram of Log-Transformed Property Prices  
**Figure 12**: Average Property Prices Over Time by Region  
**Figure 13**: Number of Property Sales Per Year  
**Figure 14**: Average Property Prices Over Time by Property Type  
**Figure 15**: Price Distribution by Property Type  
**Figure 16**: Price Distribution by Property Type (Density Line Plot)  
**Figure 17**: Monthly Average House Prices in the North West  
**Figure 18**: Time Series Decomposition of North West House Prices  
**Figure 19**: First Order Differenced Series  
**Figure 20**: ACF and PACF Plots for First-Order Differenced Series  
**Figure 21**: ACF and PACF Plots for the Seasonally Differenced Data  
**Figure 22**: SARIMA Forecast for North West House Prices (Next 3 Years)  
**Figure 23**: ARIMA vs. SARIMA Forecast Comparison  
**Figure 24**: SARIMA Model Diagnostics  
**Figure 25**: SARIMA Model: Actual vs Fitted  
**Figure 26**: Average Quarterly Property Prices  
**Figure 27**: Average Quarterly Property Prices: North vs. South  
**Figure 28**: Violin Plot for North and South Property Prices  
**Figure 29**: Average Yearly Property Price by Region **(Interactive)**